# Task 4 — Frozen CLIP Stage A (restart-safe)

Self-contained GPU workflow: restore Task 2 manifests, stage only the 2,000 required sources, persist prepared inputs, run seeds resumably, and keep `final_test` sealed.

In [6]:
# 1. GPU preflight.
import torch
assert torch.cuda.is_available(), 'Remove this Colab server and create a T4 GPU server.'
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

GPU: Tesla T4
GPU memory GB: 14.6


In [7]:
# 2. Mount Drive.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
# 3. Repository and dependencies.
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path('/content/cya-techjam26')
REPOSITORY_URL = 'https://github.com/maxi-cmyk/cya-techjam26.git'
if (PROJECT_ROOT / '.git').is_dir():
    subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_ROOT, check=True)
else:
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-colab.txt'], cwd=PROJECT_ROOT, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'], cwd=PROJECT_ROOT, check=True)
print('Repository ready:', PROJECT_ROOT)

Repository ready: /content/cya-techjam26


In [9]:
# 4. Restore frozen Task 2 contract files.
import shutil

ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts'
TASK2_ROOT = ARTIFACT_ROOT / 'task2'
DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/hackathon_data')
DRIVE_ARTIFACT_ROOT = Path('/content/drive/MyDrive/cya-techjam26/artifacts')
DRIVE_TASK2_ROOT = DRIVE_ARTIFACT_ROOT / 'task2'
DRIVE_INPUT_ARCHIVE = DRIVE_ARTIFACT_ROOT / 'task2_stagea_bundle.tar.gz'
SOURCE_IMAGES = DRIVE_DATA_ROOT / 'raw/sid_set/images'
LOCAL_SOURCE_IMAGES = Path('/content/hackathon_data/raw/sid_set/images')
GPU_CACHE_ROOT = Path('/content/clip_embedding_cache_gpu')
assert SOURCE_IMAGES.is_dir(), f'Drive SID images missing: {SOURCE_IMAGES}'
assert DRIVE_TASK2_ROOT.is_dir(), f'Drive Task 2 reports missing: {DRIVE_TASK2_ROOT}'
TASK2_ROOT.mkdir(parents=True, exist_ok=True)
for source in DRIVE_TASK2_ROOT.iterdir():
    if source.is_file() and source.suffix in {'.csv', '.json'}:
        shutil.copy2(source, TASK2_ROOT / source.name)
split_manifest = TASK2_ROOT / 'source_manifest_split.csv'
assert split_manifest.is_file(), f'Missing frozen split manifest: {split_manifest}'
print('Task 2 contract restored:', split_manifest)
print('Prepared-input archive available:', DRIVE_INPUT_ARCHIVE.is_file())

Task 2 contract restored: /content/cya-techjam26/artifacts/task2/source_manifest_split.csv
Prepared-input archive available: True


In [10]:
# 5. Restore pilots, or stage only their 2,000 required sources and rebuild.
import csv
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

if DRIVE_INPUT_ARCHIVE.is_file():
    print('Restoring prepared Task 2 inputs...')
    shutil.unpack_archive(DRIVE_INPUT_ARCHIVE, TASK2_ROOT)
candidate_root = TASK2_ROOT / 'matched_candidates'
policies = ('fixed_q96', 'uniform_q95_q100')

def candidate_counts():
    return {policy: sum(1 for path in (candidate_root / policy).rglob('*.jpg')) for policy in policies}

counts = candidate_counts()
print('Existing candidates:', counts)
if counts != {'fixed_q96': 2000, 'uniform_q95_q100': 2000}:
    with split_manifest.open(newline='', encoding='utf-8') as stream:
        rows = list(csv.DictReader(stream))
    selected = []
    selected_counts = Counter()
    for row in sorted(rows, key=lambda value: (value['label'], value['source_id'])):
        label = row['label']
        eligible = (
            row.get('eligible_for_split') == 'true'
            and row.get('duplicate_is_primary') == 'true'
            and row.get('review_required') != 'true'
            and not row.get('corruption_error')
            and row.get('c2pa_status') in {'no_manifest', 'manifest_present'}
        )
        if eligible and selected_counts[label] < 1000:
            selected.append(row)
            selected_counts[label] += 1
    assert selected_counts == {'authentic': 1000, 'ai_generated': 1000}, selected_counts
    LOCAL_SOURCE_IMAGES.mkdir(parents=True, exist_ok=True)

    def copy_source(row):
        source = SOURCE_IMAGES / Path(row['source_path']).name
        destination = Path(row['source_path'])
        assert destination.parent.resolve() == LOCAL_SOURCE_IMAGES.resolve(), destination
        assert source.is_file(), source
        if destination.is_file() and destination.stat().st_size == source.stat().st_size:
            return 'cached'
        temporary = destination.with_suffix(destination.suffix + '.part')
        shutil.copy2(source, temporary)
        temporary.replace(destination)
        return 'copied'

    print('Staging 2,000 source images...')
    results = Counter()
    errors = []
    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = {executor.submit(copy_source, row): row for row in selected}
        for completed, future in enumerate(as_completed(futures), start=1):
            try:
                results[future.result()] += 1
            except Exception as error:
                errors.append((futures[future]['source_id'], str(error)))
            if completed % 100 == 0 or completed == len(futures):
                print(f'Staging: {completed}/{len(futures)}; errors={len(errors)}')
    assert not errors, errors[:10]
    print('Staging result:', dict(results))
    subprocess.run(['make', 'task2-pilots', f'ARTIFACT_ROOT={ARTIFACT_ROOT}'], cwd=PROJECT_ROOT, check=True)
    counts = candidate_counts()
    assert counts == {'fixed_q96': 2000, 'uniform_q95_q100': 2000}, counts
    print('Persisting restart-safe Task 4 inputs...')
    local_archive = Path(shutil.make_archive('/content/task2_stagea_bundle', 'gztar', root_dir=TASK2_ROOT))
    DRIVE_INPUT_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(local_archive, DRIVE_INPUT_ARCHIVE)
    print('Saved:', DRIVE_INPUT_ARCHIVE)
print('Task 4 inputs ready:', counts)

Restoring prepared Task 2 inputs...
Existing candidates: {'fixed_q96': 2000, 'uniform_q95_q100': 2000}
Task 4 inputs ready: {'fixed_q96': 2000, 'uniform_q95_q100': 2000}


In [11]:
# 6. Pin CLIP to an immutable commit.
import json
from huggingface_hub import model_info
config_path = PROJECT_ROOT / 'configs/colab.json'
config = json.loads(config_path.read_text())
model_id = config['model']['identifier']
resolved_commit = model_info(model_id, revision=config['model']['revision']).sha
assert resolved_commit
config['model']['revision'] = resolved_commit
config_path.write_text(json.dumps(config, indent=2) + '\n')
print('Pinned:', model_id, resolved_commit)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Pinned: openai/clip-vit-large-patch14-336 ce19dc912ca5cd21c8a653c79e251e808ccabcd1


In [12]:
# 7. Resumable runner; each completed seed is immediately copied to Drive.
import os
seeds = (42, 43, 44)
physical_batch_size = 8

def run_stage_a(policy, seed):
    local_run = ARTIFACT_ROOT / 'task4' / policy / f'seed_{seed}'
    drive_run = DRIVE_ARTIFACT_ROOT / 'task4' / policy / f'seed_{seed}'
    if not (local_run / 'training_summary.json').is_file() and drive_run.is_dir():
        shutil.copytree(drive_run, local_run, dirs_exist_ok=True)
    if (local_run / 'training_summary.json').is_file():
        print(f'SKIP complete: {policy}, seed {seed}')
        return
    manifest = TASK2_ROOT / f'{policy}_manifest.csv'
    assert manifest.is_file(), manifest
    print(f'RUN: {policy}, seed {seed}', flush=True)
    subprocess.run([
        sys.executable, 'scripts/train_clip_baseline.py',
        '--manifest', str(manifest),
        '--matching-policy', policy,
        '--output-root', str(ARTIFACT_ROOT / 'task4'),
        '--cache-root', str(GPU_CACHE_ROOT),
        '--seed', str(seed),
        '--physical-batch-size', str(physical_batch_size),
    ], cwd=PROJECT_ROOT, check=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    shutil.copytree(local_run, drive_run, dirs_exist_ok=True)
    print(f'SAVED: {drive_run}')

Run one seed first. A `CLIP fixed_q96` progress bar must appear. Seeds 43/44 reuse this policy's cache.

In [ ]:
# 8. First GPU proof run.
run_stage_a('fixed_q96', 42)

RUN: fixed_q96, seed 42


In [ ]:
# 9. Inspect before launching the rest.
first_run = ARTIFACT_ROOT / 'task4/fixed_q96/seed_42'
extraction = json.loads((first_run / 'extraction_report.json').read_text())
training = json.loads((first_run / 'training_summary.json').read_text())
print(json.dumps({'extraction': extraction, 'best_clean_accuracy': training['best_values']['clean'], 'epochs_completed': training['epochs_completed']}, indent=2))

In [ ]:
# 10. Remaining runs; completed runs are skipped.
for policy in policies:
    for seed in seeds:
        run_stage_a(policy, seed)

In [ ]:
# 11. Clean-selection Task 5 reports.
for policy in policies:
    for seed in seeds:
        predictions = ARTIFACT_ROOT / 'task4' / policy / f'seed_{seed}' / 'best_clean_predictions.csv'
        output = ARTIFACT_ROOT / 'task5' / policy / f'seed_{seed}'
        subprocess.run([sys.executable, 'scripts/evaluate_predictions.py', '--predictions', str(predictions), '--output', str(output)], cwd=PROJECT_ROOT, check=True)
        shutil.copytree(output, DRIVE_ARTIFACT_ROOT / 'task5' / policy / f'seed_{seed}', dirs_exist_ok=True)
print('Task 5 clean reports saved')

In [ ]:
# 12. Multi-seed matching-policy comparison.
comparison_path = ARTIFACT_ROOT / 'task4/policy_comparison.json'
subprocess.run([sys.executable, 'scripts/compare_stage_a.py', '--task4-root', str(ARTIFACT_ROOT / 'task4'), '--output', str(comparison_path)], cwd=PROJECT_ROOT, check=True)
drive_comparison = DRIVE_ARTIFACT_ROOT / 'task4/policy_comparison.json'
drive_comparison.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(comparison_path, drive_comparison)
comparison = json.loads(comparison_path.read_text())
print('Selected clean-only policy:', comparison['selected_policy'])
print('Saved:', drive_comparison)